# DDPF Description Module Minimal Demo

This notebook tests the reusable `ddpf.description` library.

It shows two enhancement modes:

1. **Manual enhancement**, following the original thesis logic, by importing existing domain/network ontologies and aligning them with PropaPhen.
2. **LLM-assisted enhancement**, using an abstract LLM interface and the Ollama implementation.

The notebook avoids raw thesis data and only uses ontology files already kept in `data/ontologies/`.


In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path

from ddpf.description import DescriptionModule
from ddpf.llm import OllamaLLMClient

## Paths

In [4]:
ROOT = Path("../..").resolve()

PROPAPHEN_PATH = ROOT / "data" / "ontologies" / "PropaPhen" / "PropaPhen.owl"
UMLS_ONTOLOGY_PATH = ROOT / "data" / "ontologies" / "propaphenplus" / "saved" / "umlsonto.owl"
WORLDKG_ONTOLOGY_PATH = ROOT / "data" / "ontologies" / "propaphenplus" / "saved" / "worldkg.owl"

OUTPUT_DIR = ROOT / "data" / "ontologies" / "propaphenplus" / "generated"
MANUAL_OUTPUT_PATH = OUTPUT_DIR / "propaphenplus_manual.owl"
LLM_OUTPUT_PATH = OUTPUT_DIR / "propaphenplus_llm.owl"

for path in [PROPAPHEN_PATH, UMLS_ONTOLOGY_PATH, WORLDKG_ONTOLOGY_PATH]:
    print(path, "exists:", path.exists())

C:\Users\henri\Documents\git\personal\DDPF-Health-Risks\data\ontologies\PropaPhen\PropaPhen.owl exists: True
C:\Users\henri\Documents\git\personal\DDPF-Health-Risks\data\ontologies\propaphenplus\saved\umlsonto.owl exists: True
C:\Users\henri\Documents\git\personal\DDPF-Health-Risks\data\ontologies\propaphenplus\saved\worldkg.owl exists: True


## Manual ontology enhancement

In [5]:
description = DescriptionModule(
    propaphen_path=PROPAPHEN_PATH,
    domain_ontology_path=UMLS_ONTOLOGY_PATH,
    network_ontology_path=WORLDKG_ONTOLOGY_PATH,
)

description.load(load_owlready=True)

manual_output = description.enhance_manual(
    output_path=MANUAL_OUTPUT_PATH
)

manual_output

WindowsPath('C:/Users/henri/Documents/git/personal/DDPF-Health-Risks/data/ontologies/propaphenplus/generated/propaphenplus_manual.owl')

## Optional LLM-assisted enhancement

This cell requires Ollama to be running locally.

Example terminal command:

```bash
ollama run llama3.1
```

Then set `RUN_LLM = True`.


In [8]:
RUN_LLM = True

if RUN_LLM:
    llm = OllamaLLMClient(model="llama3.1")

    text = '''
    Reports mention a suspicious respiratory outbreak with fever, cough,
    hospitalization, international travel, and confirmed cases in several cities.
    '''

    llm_ontology = description.enhance_with_llm(
        llm=llm,
        text=text,
        parent_class_name="Phenomenon",
        max_concepts=5,
        output_path=LLM_OUTPUT_PATH,
    )

    print("LLM-enhanced ontology saved to:", LLM_OUTPUT_PATH)
    print("LLM-enhanced ontology saved to:", llm_ontology)

    llm_output_path = LLM_OUTPUT_PATH.with_suffix(".llm.txt")

    if llm_output_path.exists():
        print("\nLLM suggested concepts:")
        print(llm_output_path.read_text(encoding="utf-8"))
else:
    print("LLM enhancement skipped. Set RUN_LLM = True to test with Ollama.")

LLM-enhanced ontology saved to: C:\Users\henri\Documents\git\personal\DDPF-Health-Risks\data\ontologies\propaphenplus\generated\propaphenplus_llm.owl
LLM-enhanced ontology saved to: C:\Users\henri\Documents\git\personal\DDPF-Health-Risks\data\ontologies\propaphenplus\generated\propaphenplus_llm.owl

LLM suggested concepts:
Outbreak
Respiratory Outbreak
Fever
Hospitalization
International Travel


## Minimal import check

In [9]:
import ddpf
from ddpf import description as description_package

print("ddpf version:", ddpf.__version__)
print("Description API:", description_package.__all__)

ddpf version: 0.1.0
Description API: ['DescriptionModule', 'ManualOntologyEnhancer', 'load_ontology_graph', 'load_ontology_with_owlready', 'save_graph']
